# IoT-Engine Pipeline Demo
Replicates the full disabled IoT-engine flow:

```
User query
  → Vector DB  (top-3 service type names)
  → MongoDB    (geo-search within radius)
  → Recommender (travel time · occupancy · overall service time)
  → Generator  (Groq LLM → human-readable response)
```

## Setup · Install Dependencies
Run this cell once when setting up a new environment. Skip if packages are already installed.

In [8]:
%pip install -q \
    "pymilvus[milvus_lite]>=2.5.0,<2.6.0" \
    "numpy<2" \
    "langchain==0.3.1" \
    "langchain-core==0.3.6" \
    "langchain-groq==0.2.0" \
    "pymongo==4.9.1" \
    "geopy==2.4.1" \
    "openrouteservice==2.3.3" \
    "python-dotenv==1.0.1" \
    "pandas" \
    "sentence-transformers"

print("All dependencies installed.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jina 3.34.0 requires grpcio<=1.68.0,>=1.46.0, but you have grpcio 1.80.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
All dependencies installed.


In [9]:
import sys, os
from dotenv import load_dotenv

# Point Python at backend/src so all internal modules resolve
BACKEND_SRC = os.path.abspath(os.path.join(os.getcwd(), "..", "backend", "src"))
if BACKEND_SRC not in sys.path:
    sys.path.insert(0, BACKEND_SRC)

# Load researcher credentials from research/credentials.env
ENV_PATH = os.path.abspath(os.path.join(os.getcwd(), "credentials.env"))
load_dotenv(ENV_PATH, override=True)

# Point vector DB module at the local copy of milvus_lite.db
MILVUS_DB_PATH = os.path.abspath(os.path.join(os.getcwd(), "milvus_lite.db"))
os.environ["MILVUS_DB_PATH"] = MILVUS_DB_PATH

print("BACKEND_SRC  :", BACKEND_SRC)
print("ENV loaded   :", ENV_PATH)
print("MILVUS_DB    :", MILVUS_DB_PATH)
print("MONGODB_URL  :", os.environ.get("MONGODB_URL", "")[:40], "...")
print("GROQ_API_KEY :", os.environ.get("GROQ_API_KEY", "")[:15], "...")

BACKEND_SRC  : /Users/elewah/Desktop/repo/localelive/backend/src
ENV loaded   : /Users/elewah/Desktop/repo/localelive/research/credentials.env
MILVUS_DB    : /Users/elewah/Desktop/repo/localelive/research/milvus_lite.db
MONGODB_URL  : mongodb+srv://ashraf:GAYe3coK10tBmSDu@cl ...
GROQ_API_KEY : gsk_0EUqaxpKOvy ...


## Step 0 · User Query & Location
Edit these two cells — everything downstream updates automatically.

In [10]:
# ── EDIT HERE ──────────────────────────────────────────────────
USER_QUERY = "I need a coffee shop"

# User GPS coordinates  (default: downtown Toronto)
LATITUDE  = 43.6532
LONGITUDE = -79.3832

SEARCH_RANGE_M = 10_000   # MongoDB geo-search radius in metres
VECTOR_TOP_K   = 3        # how many service types to retrieve
MONGO_LIMIT    = 5        # how many nearby places to retrieve
# ───────────────────────────────────────────────────────────────
print(f'Query    : "{USER_QUERY}"')
print(f'Location : {LATITUDE}, {LONGITUDE}')

Query    : "I need a coffee shop"
Location : 43.6532, -79.3832


## Step 1 · Vector DB — Top-K Service Type Names

In [11]:
from vector_db.vector_database import vector_search, vector_db_push_batch

# Ensure index is built
vector_db_push_batch()

service_types = vector_search(USER_QUERY, limit=VECTOR_TOP_K)
print(f"Top {VECTOR_TOP_K} matched service types:", service_types)

# IoT-engine always searches the #1 match in MongoDB
primary_collection = service_types[0]
print(f"\nPrimary MongoDB collection → '{primary_collection}'")

Top 3 matched service types: ['coffee shop', 'convenience store', 'deli shop']

Primary MongoDB collection → 'coffee shop'


## Step 2 · MongoDB — Geo-search for Nearby Places

In [12]:
import pandas as pd
from mongo_db.database_connection import get_nearByPlaces

raw_cursor = get_nearByPlaces(
    latitude=LATITUDE,
    longitude=LONGITUDE,
    collection_name=primary_collection,
    search_range=SEARCH_RANGE_M,
    limit=MONGO_LIMIT,
)
services = list(raw_cursor)
print(f"{len(services)} places found within {SEARCH_RANGE_M/1000:.0f} km\n")

preview_cols = ["Service Name", "Service Address", "Rate"]
df_raw = pd.DataFrame(
    [{c: s.get(c) for c in preview_cols} for s in services]
)
df_raw

5 places found within 10 km



,Service Name,Service Address,Rate
0,Timothy's World Coffee,"Bell Trinity Square, 483 Bay St, Toronto, ON M...",4.4
1,Mos Mos Coffee,"Thomson Building - Path, 65 Queen St W, Toront...",4.7
2,Dispatch Coffee,"390 Bay St., Toronto, ON M5H 2Y2, Canada",4.6
3,Starbucks,"176 Yonge St, Toronto, ON M5C 2L7, Canada",3.8
4,Second Cup Café,"1 Dundas St W Unit D102, Toronto, ON M5G 1Z3, ...",4.1


## Step 3 · Service Recommender — Travel Time · Occupancy · Overall Score

In [13]:
# get_recommendedSerivce mutates the list in-place, so work on a copy
import copy
from serivce_recommender.sorting_serivces import get_recommendedSerivce

services_copy = copy.deepcopy(services)
ranked = get_recommendedSerivce(LONGITUDE, LATITUDE, services_copy)

df_ranked = pd.DataFrame(ranked)
display_cols = [
    c for c in [
        "Service Name", "Service Address", "Rate",
        "Occupancy", "Estimated Travel Time (min)", "Estimated Overall Service Time (min)"
    ]
    if c in df_ranked.columns
]
print("Ranked results (lower Overall Service Time = better):")
df_ranked[display_cols].sort_values("Estimated Overall Service Time (min)")

Ranked results (lower Overall Service Time = better):


,Service Name,Service Address,Rate,Occupancy,Estimated Travel Time (min),Estimated Overall Service Time (min)
1,Mos Mos Coffee,"Thomson Building - Path, 65 Queen St W, Toront...",4.7,1,1.2,6.2
0,Timothy's World Coffee,"Bell Trinity Square, 483 Bay St, Toronto, ON M...",4.4,1,1.3,6.3
2,Dispatch Coffee,"390 Bay St., Toronto, ON M5H 2Y2, Canada",4.6,1,1.3,6.3
3,Starbucks,"176 Yonge St, Toronto, ON M5C 2L7, Canada",3.8,1,2.9,7.9
4,Second Cup Café,"1 Dundas St W Unit D102, Toronto, ON M5G 1Z3, ...",4.1,68,3.3,343.3


## Step 4 · Generator Agent — Human-Readable Response via Groq

In [14]:
from langchain_core.messages import SystemMessage, HumanMessage
from utils import llm
from agents_prompt import IoT_engine_prompt

context_json = str(ranked)

prompt = [
    SystemMessage(content=IoT_engine_prompt.format(
        JsonObject=context_json,
        node="IoT_engine"
    )),
    HumanMessage(content=USER_QUERY),
]

response = llm.invoke(prompt)

print("═" * 60)
print(response.content)
print("═" * 60)

════════════════════════════════════════════════════════════
If you’re looking for a quick, low‑crowd coffee spot, Mos Mos Coffee is a great pick. It’s just a minute‑and‑a‑half away, takes about six minutes to get your order, and has a solid 4.7‑star rating.

Dispatch Coffee is another solid choice—same travel time, a 4.6 rating, and a similar quick service window.

Timothy’s World Coffee is very close too, with a 4.4 rating and a quick turnaround.

If you’re willing to walk a bit farther, Starbucks offers a familiar experience, though it’s a bit slower (about eight minutes) and a bit farther (almost three minutes).

All of these options have low occupancy, so you’re unlikely to run into a crowd.

Data source is the IoT_engine agent.
════════════════════════════════════════════════════════════


## Bonus · Try Different Queries Quickly

In [15]:
import copy

def run_pipeline(query: str, lat: float, lng: float,
                 search_range: int = 10_000,
                 vector_k: int = 3, mongo_limit: int = 5) -> str:
    """End-to-end IoT pipeline in one call."""
    # 1. Vector search
    svc_types = vector_search(query, limit=vector_k)
    collection = svc_types[0]

    # 2. MongoDB geo-search
    raw = list(get_nearByPlaces(lat, lng, collection, search_range, mongo_limit))
    if not raw:
        return f"No places found for '{collection}' within {search_range/1000:.0f} km."

    # 3. Recommender
    ranked_results = get_recommendedSerivce(lng, lat, copy.deepcopy(raw))

    # 4. Generator
    prompt = [
        SystemMessage(content=IoT_engine_prompt.format(
            JsonObject=str(ranked_results), node="IoT_engine"
        )),
        HumanMessage(content=query),
    ]
    resp = llm.invoke(prompt)
    return resp.content


# ── Try these ────────────────────────────────────────────────────
test_queries = [
    "I need a pharmacy",
    "gym to work out",
    "somewhere to get pizza",
]

for q in test_queries:
    print(f'\n>>> Query: "{q}"')
    print(run_pipeline(q, LATITUDE, LONGITUDE))
    print("-" * 60)


>>> Query: "I need a pharmacy"
If you’re looking for a quick, convenient pharmacy with a good reputation, Rexall at 120 Adelaide St W in Toronto is a solid choice. It’s only about a 3‑minute drive, has a short average service time of roughly 8 minutes, and a high rating of 4.1. The pharmacy is usually not crowded, so you’ll likely get through quickly.

If you’d like a close alternative, Shoppers Drug Mart on Bay St. is just a 3‑minute walk away, offers a similar service time, and has a respectable rating of 3.8.

Both locations are well‑rated, low‑traffic, and close by, making them ideal for a quick pharmacy visit.

Data source: IoT_engine agent.
------------------------------------------------------------

>>> Query: "gym to work out"
**Equinox Bay Street**  
- **Address:** 199 Bay St., Toronto, ON M5L 1E2, Canada  
- **Rate:** 3.8 ★  
- **Occupancy:** 1 (very low)  
- **Estimated Travel Time:** 2.3 min  
- **Estimated Overall Service Time:** 7.3 min  

This gym offers a very low cro

I0712 17:45:17.681249  281069 chttp2_transport.cc:1369] unix:/var/folders/p6/92s1vm8971s6k22vcrw2157r0000gn/T/tmp3rn0oq5f_milvus_lite.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11, grpc_status:14}
E0712 17:45:17.681682  281069 chttp2_transport.cc:1401] unix:/var/folders/p6/92s1vm8971s6k22vcrw2157r0000gn/T/tmp3rn0oq5f_milvus_lite.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 20000ms
